# Fine-tuning：Twinkle AI `gemma-3-4B-T1-it` × Unsloth QLoRA

## 這份 notebook 在做什麼

把信義鄉相關知識**直接微調進模型權重**，讓模型之後能不靠檢索、直接回答「信義鄉在日治時期發生過什麼事」這類問題。目的不是複製 `classify_chain.py`／`classify_agent.py` 的分類任務。

跟 `src/rag/query_engine.py` 的 RAG 問答互為對照：

| | RAG（`query_engine.py`） | Fine-tuning（這份 notebook） |
|---|---|---|
| 知識放在哪 | 外部語料庫，推論時檢索 | 直接寫進模型權重 |
| 回答時 | 會檢索＋附上來源 | 不檢索、不附來源 |

## 訓練資料怎麼來

1. **生成合成 QA 對**：`generate_qa.py` 呼叫 Gemini（`gemini-3.1-flash-lite`）對每個已分類段落生成問答對
   - 實測比較過 Claude／Gemini／Groq 三家後選定，詳見 README
   - 執行：`python -m src.finetune.generate_qa --all`
2. **切分 train/eval**：`prepare_dataset.py` 把 QA 對切成訓練集與 held-out 評估集
   - 執行：`python -m src.finetune.prepare_dataset`

## 基底模型

**`twinkle-ai/gemma-3-4B-T1-it`**（Twinkle AI，台灣語境 fine-tuned 開源模型，Gemma 3 4B）

- 訓練重點是台灣人文／地方文史語境，跟本專案內容最貼題
- HF 上需先接受 gated license（見 Step 1）

> 原本也試過 `twinkle-ai/Llama-3.2-3B-F1-Instruct`，但它用 Hermes 格式訓練、chat template 跟標準格式不同，風險較高，已改成只用 Gemma（Google 官方標準 chat template）。

## 執行環境

- RTX 5060 Laptop GPU，8GB VRAM
- 4-bit QLoRA（Unsloth）微調

## 前置作業

- [ ] `pip install -r requirements-finetune.txt`（torch 依本機 CUDA 版本另外裝，見檔案內的安裝順序說明）
- [ ] `python -m src.finetune.generate_qa --all`（會呼叫 Gemini API 產生費用，執行前會先看到花費估算）
- [ ] `python -m src.finetune.prepare_dataset`（切出 `data/train.jsonl`／`data/eval_ids.json`）

## 用法

前置作業都完成後，從頭到尾逐格執行一次，就會產出一個 LoRA adapter（存在 `adapters/gemma/`）。


## Step 0：環境檢查

先確認 PyTorch 吃得到 GPU、VRAM 夠不夠，避免後面步驟跑到一半才發現環境有問題。

**這裡也修兩個本機環境特有的坑**（在 `import torch` 之前處理，實測跑過 Step 6 才找出來的）：
- Windows 帳號名稱是中文，且系統 ANSI codepage 讀出來的位元組不是合法 UTF-8。Triton 的 C extension（`triton._C.libtriton.getenv`）內部讀環境變數會用 UTF-8 解碼，讀到帳號名稱就會 `UnicodeDecodeError` 炸掉，只在偵測到 `USERNAME` 不是純 ASCII 時才覆寫成安全值繞過，其他帳號本來就是 ASCII 安全就不動它，避免影響 torch/unsloth 依 USERNAME 組出來的快取路徑。
- Gemma3 的滑動視窗注意力用 PyTorch 的 FlexAttention 實作，會透過 `torch.compile`／TorchInductor 即時編譯 Triton kernel，但編譯出的 kernel 需要的 shared memory 超過這張消費級 GPU 的硬體上限（`OutOfMemoryError: out of resource`）。直接關掉 `torch.compile`，退回不編譯的標準路徑。

In [ ]:
import os

# Windows 帳號的 USERNAME 如果不是純 ASCII（例如中文），系統目前的 ANSI codepage 讀出來的位元組
# 就不是合法 UTF-8。Triton 的 C extension（triton._C.libtriton.getenv）內部讀環境變數時會用
# UTF-8 解碼，讀到這種 USERNAME 就會 UnicodeDecodeError 炸掉。只在偵測到非 ASCII 時才覆寫
# （避免在其他帳號本來就是 ASCII 安全的機器上，把 USERNAME 誤蓋成這裡寫死的替代值，影響
# torch/unsloth 依 USERNAME 組出來的快取路徑）。在 import torch 之前處理。
_username = os.environ.get("USERNAME", "")
if not _username.isascii():
    os.environ["USERNAME"] = "user"

# Gemma3 的滑動視窗注意力用 FlexAttention 實作，會透過 torch.compile/TorchInductor 即時編譯
# Triton kernel，但編譯出的 kernel 需要的 shared memory 超過這張消費級 GPU 的硬體上限
# （OutOfMemoryError: out of resource）。直接關掉 torch.compile，退回不編譯的標準路徑。
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import torch

print("CUDA 可用：", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU：", torch.cuda.get_device_name(0))
    print(f"總 VRAM：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("沒有偵測到 CUDA GPU，請先確認驅動與 torch 安裝（見 requirements-finetune.txt 的說明）。")

## Step 1：設定模型與路徑

`gemma-3-4B-T1-it` 需要先到 [Hugging Face](https://huggingface.co/twinkle-ai/gemma-3-4B-T1-it) 接受 gated license，並在專案根目錄的 `.env` 補上 `HF_TOKEN=hf_xxxx`。

In [2]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "finetune" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.finetune.prepare_dataset import MODEL_NAME

MAX_SEQ_LENGTH = 1536  # 系統提示（分類清單＋篇章參考）＋段落＋JSON 輸出，估計上限

print("ROOT：", ROOT)
load_dotenv(ROOT / ".env")
if not os.getenv("HF_TOKEN"):
    raise RuntimeError("gemma 是 gated model，請先在 .env 補上 HF_TOKEN（見上方說明）。")

ADAPTER_DIR = ROOT / "src" / "finetune" / "adapters" / "gemma"
DATA_DIR = ROOT / "src" / "finetune" / "data"
print("本次微調模型：", MODEL_NAME)
print("adapter 輸出目錄：", ADAPTER_DIR)

ROOT： D:\Xinyi_Township_Gazetteer
本次微調模型： twinkle-ai/gemma-3-4B-T1-it
adapter 輸出目錄： D:\Xinyi_Township_Gazetteer\src\finetune\adapters\gemma


"## Step 2：讀取訓練資料\n",
"\n",
"讀 `prepare_dataset.py` 產出的 `train.jsonl`（每筆是 system/user/assistant 三則訊息，user 是合成問題、assistant 是合成答案），印一筆出來看實際訓練資料長什麼樣子。"

In [3]:
import json

train_path = DATA_DIR / "train.jsonl"
if not train_path.exists():
    raise FileNotFoundError(f"找不到 {train_path}，請先執行：python -m src.finetune.prepare_dataset")

raw_examples = [json.loads(line) for line in train_path.open(encoding="utf-8") if line.strip()]
print(f"訓練樣本數：{len(raw_examples)}")
print()
sample = raw_examples[0]
for m in sample["messages"]:
    preview = m["content"][:120].replace("\n", " ")
    print(f"[{m['role']}] {preview}...")

訓練樣本數：9848

[system] 你是熟悉《南投縣信義鄉志》相關知識的在地知識助理，請根據你所學到的信義鄉歷史、地理、社會、文化、產業等知識，用完整、正確的中文直接回答使用者的問題。...
[user] 布農族的八部合音展現了什麼樣的文化精神？...
[assistant] 布農族的八部合音不僅是極具生命力與靈性的祈禱方式，更深刻體現了族人對大自然的崇敬與愛護。在進行合音時，族人會刻意壓抑個人的表現，全心全意地追求群體的和諧與合作，以傳達對大自然的禮讚。這種謹慎遵守時節與禁忌的態度，不僅彰顯了祖先流傳下來的智慧...


## Step 3：載入 4-bit 基底模型

用 Unsloth 的 `FastLanguageModel` 以 4-bit 量化載入基底模型——這是能在 8GB VRAM 上塞得下 3B/4B 模型的關鍵。

In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    token=os.getenv("HF_TOKEN"),
)
print(model.config)

d:\Xinyi_Township_Gazetteer\.venv\Lib\site-packages\unsloth\__init__.py:1427: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0715 10:53:26.738000 2588 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead

🦥 Unsloth Zoo will now patch everything to make training faster!


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

==((====))==  Unsloth 2026.7.2: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5060 Laptop GPU. Num GPUs = 1. Max memory: 7.96 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/445 [00:00<?, ?it/s]

d:\Xinyi_Township_Gazetteer\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Gemma3TextConfig {
  "_sliding_window_pattern": 6,
  "architectures": [
    "Gemma3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "attn_logit_softcapping": null,
  "bos_token_id": 2,
  "cache_implementation": "hybrid",
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "final_logit_softcapping": null,
  "head_dim": 256,
  "hidden_activation": "gelu_pytorch_tanh",
  "hidden_size": 2560,
  "initializer_range": 0.02,
  "intermediate_size": 10240,
  "layer_types": [
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attent

## Step 4：掛上 LoRA

LoRA 只在原模型旁掛一小組可訓練參數，凍結原本的權重——這就是為什麼 8GB VRAM 也能微調一個 3B/4B 模型。下面會印出可訓練參數量佔全模型的比例，親眼確認這件事。

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
trainable params: 29,802,496 || all params: 3,910,065,664 || trainable%: 0.7622


## Step 5：組成 SFTTrainer 用的資料集

- 用模型自帶的 chat template（`tokenizer.apply_chat_template`）把 system/user/assistant 三則訊息組成模型認得的完整字串
  - 如果印出來的格式看起來不對，代表這個模型的 tokenizer 沒有內建正確的 chat template，需要對照模型卡另外指定
- 額外組一份 **held-out 評估集**（`eval_ids.json` 對應的段落，訓練時完全不會用到），格式跟訓練資料一致
  - 供 Step 7 訓練時同步監控 held-out loss，不用等 3 個 epoch 全部跑完才知道有沒有過擬合


In [6]:
from datasets import Dataset

from src.finetune.prepare_dataset import QA_CACHE_PATH, _to_chat_examples


def to_text(example: dict) -> dict:
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)}


full_dataset = Dataset.from_list(raw_examples).map(to_text)
print(full_dataset[0]["text"][:500])

# held-out 評估集：跟 prepare_dataset.py 產生 train.jsonl 用同一份 _to_chat_examples()，
# 只是這裡篩的是 eval_ids（訓練從沒看過的段落），格式才會跟訓練資料完全一致可比。
eval_ids = set(json.loads((DATA_DIR / "eval_ids.json").read_text(encoding="utf-8")))
eval_examples = []
with QA_CACHE_PATH.open(encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        if record["id"] in eval_ids:
            eval_examples.extend(_to_chat_examples(record["id"], record["pairs"]))

eval_dataset = Dataset.from_list(eval_examples).map(to_text)
print(f"held-out 評估集：{len(eval_dataset)} 組 QA")

Map:   0%|          | 0/9848 [00:00<?, ? examples/s]

<bos><start_of_turn>user
你是熟悉《南投縣信義鄉志》相關知識的在地知識助理，請根據你所學到的信義鄉歷史、地理、社會、文化、產業等知識，用完整、正確的中文直接回答使用者的問題。

布農族的八部合音展現了什麼樣的文化精神？<end_of_turn>
<start_of_turn>model
布農族的八部合音不僅是極具生命力與靈性的祈禱方式，更深刻體現了族人對大自然的崇敬與愛護。在進行合音時，族人會刻意壓抑個人的表現，全心全意地追求群體的和諧與合作，以傳達對大自然的禮讚。這種謹慎遵守時節與禁忌的態度，不僅彰顯了祖先流傳下來的智慧，更說明了布農族人如何在生活中與自然環境連結，展現出高度的敬畏與生命價值。<end_of_turn>



Map:   0%|          | 0/152 [00:00<?, ? examples/s]

held-out 評估集：152 組 QA


## Step 6：Smoke test（小子集，先驗證工具鏈）

Unsloth／bitsandbytes 在原生 Windows 上的相容性不像 Linux 穩定，先用一小部分資料（300 筆、1 epoch）跑一次，確認能訓練、能存檔、顯存沒有 OOM——讓環境問題在這裡先曝露出來，而不是 full run 跑到一半才發現。

- **這一格印出的耗時是這張顯卡上第一個真實數字**：拿它去換算下一步全量訓練大概要多久，比事先用文字猜測準得多
- **這一格訓練出來的模型不會拿去正式使用**：Step 7 會先跑 Step 6.5 重新載入一份乾淨的模型再訓練，避免這裡的 300 筆額外訓練污染正式的 adapter
- **執行前務必核對**：這裡第一次用到 `train_on_responses_only`（只對 assistant 回答算 loss，不浪費訓練訊號在系統提示與問題上）。看 Step 5 印出的 `full_dataset[0]["text"]`，assistant 回合開頭是不是 `<start_of_turn>model\n`（Gemma 標準格式）——不是的話要把下面的 `response_part` 改成實際看到的字串，否則遮罩會抓錯位置


In [7]:
import time

from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

smoke_dataset = full_dataset.select(range(min(300, len(full_dataset))))

smoke_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=smoke_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        seed=42,
        padding_free=False,  # Windows 非 UTF-8 locale 下，auto-enable 的 padding-free 會觸發 TorchInductor 編譯，讀檔時用系統預設編碼（cp950）解碼 UTF-8 內容而炸掉；關掉繞過這條編譯路徑
        output_dir=str(ADAPTER_DIR / "_smoke_test"),
        report_to="none",
    ),
)
smoke_trainer = train_on_responses_only(
    smoke_trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

start = time.time()
smoke_trainer.train()
elapsed = time.time() - start
print(f"\nSmoke test（{len(smoke_dataset)} 筆、1 epoch）耗時：{elapsed:.1f} 秒")
print(f"換算全量訓練（{len(full_dataset)} 筆）粗估每 epoch：約 {elapsed / len(smoke_dataset) * len(full_dataset) / 60:.1f} 分鐘")

Unsloth: Tokenizing ["text"]:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 106}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 1 | Total steps = 38
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,802,496 of 3,910,065,664 (0.76% trained)


InductorError: UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa6 in position 48: invalid start byte

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"


## Step 6.5：重新載入乾淨模型（避免 smoke test 汙染正式訓練）

Step 6 的 `model` 已經被 300 筆資料訓練過 1 epoch，如果直接拿去做 Step 7 的正式訓練，最終 adapter 等於「全量 3 epoch + 前 300 筆多 1 次」，而且這 300 筆剛好是 `train.jsonl` 裡排最前面的段落（`prepare_dataset.py` 沒有打亂順序），會造成跟段落順序相關的過擬合偏差。

這裡重新執行一次 Step 3＋4 的邏輯，拿到一份沒被動過的乾淨模型，`print_trainable_parameters()` 的輸出應該要跟 Step 4 第一次一模一樣——如果不一樣，代表沒有重新載入成功。

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    token=os.getenv("HF_TOKEN"),
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

## Step 7：正式訓練（全量資料）

確認 Step 6 的 smoke test 沒問題、且已經跑過 Step 6.5 拿到乾淨模型後，才執行這格。

- 用 Step 6 印出的換算時間評估要不要調整 `NUM_EPOCHS`
- 這一格會跑比較久，過程中會逐步印出訓練 loss
- 每個 epoch 結束後也會印出 held-out（`eval_dataset`）的 loss，能直接看到訓練是怎麼收斂的、有沒有開始過擬合


In [ ]:
NUM_EPOCHS = 3  # 依 Step 6 的時間換算調整

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=full_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        logging_steps=20,
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        seed=42,
        padding_free=False,  # 同 Step 6：關掉避免 Windows 非 UTF-8 locale 觸發 TorchInductor 編譯報 UnicodeDecodeError
        eval_strategy="epoch",
        output_dir=str(ADAPTER_DIR / "_checkpoints"),
        save_strategy="epoch",
        save_total_limit=2,
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

start = time.time()
train_result = trainer.train()
elapsed = time.time() - start
print(f"\n全量訓練耗時：{elapsed / 60:.1f} 分鐘")
print(train_result.metrics)

## Step 8：存 adapter

In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f"adapter 已存到：{ADAPTER_DIR}")

"## Step 9：inline 快速測試\n",
"\n",
"對幾筆 held-out 段落生成的問題（`eval_ids.json` 對應到 `qa_pairs.jsonl` 裡的 QA 對，訓練時完全沒看過）做推論，肉眼比對模型答案跟合成參考答案，再進到 `evaluate.py` 做完整的並列比對報告。"

In [ ]:
from src.finetune.prepare_dataset import FT_SYSTEM_PROMPT

FastLanguageModel.for_inference(model)

qa_records = {r["id"]: r["pairs"] for r in (json.loads(l) for l in (DATA_DIR / "qa_pairs.jsonl").open(encoding="utf-8") if l.strip())}

shown = 0
for eid in eval_ids:
    for pair in qa_records.get(eid, []):
        messages = [
            {"role": "system", "content": FT_SYSTEM_PROMPT},
            {"role": "user", "content": pair["question"]},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        output_ids = model.generate(**inputs, max_new_tokens=300, do_sample=False)
        generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        print(f"=== {eid} ===")
        print("問題：", pair["question"])
        print("模型答案：", generated.strip()[:300])
        print("參考答案：", pair["answer"])
        print()
        shown += 1
    if shown >= 5:
        break